# Handling the datos faltantes y penalización de datos erróneos.

Notebook para el ensayo de técnicas de imputación en el manejo de nulos y para la correción de imputaciones metodológics.

In [1]:
import os
from pathlib import Path
print("Directorio actual:", os.getcwd())

current_dir = Path.cwd()
project_root = current_dir
while not (project_root / '.git').exists() and project_root != project_root.parent:
    project_root = project_root.parent
    os.chdir("..")

print(f"Raíz del proyecto: {project_root}")
print("CWD cambiado a raíz del proyecto")

Directorio actual: /home/ferrus/Documents/university/semester_x/MA2003B/PROYECTO_MA2003B/notebooks/02_data_correction
Raíz del proyecto: /home/ferrus/Documents/university/semester_x/MA2003B/PROYECTO_MA2003B
CWD cambiado a raíz del proyecto


## Imports

In [2]:
import pandas as pd
import numpy as np

path_master_df = "data/processed/master_table.csv"
df_master = pd.read_csv(path_master_df)

/tmp/ipykernel_4239/2896344593.py:5: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_master = pd.read_csv(path_master_df)


## Reorganización de etiquetas para estaciones

In [3]:
def clean_column_pandas(df: pd.DataFrame, column_name: str) -> pd.DataFrame:
    """Map column removing whitespace and converting to lowercase using pandas"""
    df[column_name] = df[column_name].str.replace(' ', '').str.lower()
    return df

set_estaciones = set(df_master["estacion"])


df_master_v2 = clean_column_pandas(df = df_master,
                                   column_name = "estacion")

set_estaciones_v2 = set(df_master_v2['estacion'])

print(pd.Series(list(set_estaciones_v2)))

0      suroeste
1           sur
2      noreste2
3        norte2
4       noreste
5      noroeste
6      sureste2
7      noreste3
8     noroeste3
9      sureste3
10      sureste
11    suroeste2
12       centro
13        norte
14    noroeste2
dtype: object


## Filtrado inteligente

In [4]:
variables_contaminantes = ['co', 'no', 'no2', 'nox',
                           'o3', 'pm10', 'pm2.5', 'prs',
                           'rainf', 'rh', 'so2', 'sr',
                           'tout', 'wsr', 'wdr']

nombres_estaciones      = list(set_estaciones_v2)

def filter_variable_station(df: pd.DataFrame, variable : str, station : str)-> pd.DataFrame:
    """
    Returns the complete time series of the variable for the specified station
    """
    df_c = df.copy()
    df_c = df_c[df_c['estacion'] == station].reset_index()
    df_c = df_c[["date_index", variable]]
    return df_c

df_co_suroeste = filter_variable_station(df_master_v2, "co", "suroeste")

In [5]:
df_co_suroeste

,date_index,co
0,2022-01-01 00:00:00,NaN
1,2022-01-01 01:00:00,3.28
2,2022-01-01 02:00:00,3.25
3,2022-01-01 03:00:00,3.02
4,2022-01-01 04:00:00,2.73
...,...,...
26297,2024-12-31 19:00:00,2.17
26298,2024-12-31 20:00:00,2.38
26299,2024-12-31 21:00:00,2.50
26300,2024-12-31 22:00:00,2.54


## Visualización de datos interanuales

A partir de la organización de la tabla queremos visualizar el cambio interanual de los valores medidos. Para esto es necesario separar los registros tabulares por su año asignado. Luego analizamos la consistencia interanual de los registros 

In [6]:
from typing import Optional, Dict

def split_timeseries_by_years(df: pd.DataFrame, 
                             date_col: str = "date_index", 
                             value_col: Optional[str] = None) -> Dict[int, pd.DataFrame]:
    """
    Split hourly time series into separate DataFrames by year
    
    Args:
        df: DataFrame with datetime and value columns
        date_col: Name of date column
        value_col: Name of value column (if None, uses second column)
    
    Returns:
        Dictionary with year as key and DataFrame as value
    """
    df_work = df.copy()
    
    if value_col is None:
        value_col = df_work.columns[1]
    
    df_work[date_col] = pd.to_datetime(df_work[date_col])
    df_work['year'] = df_work[date_col].dt.year
    
    year_dfs = {}
    for year in sorted(df_work['year'].unique()):
        year_df = df_work[df_work['year'] == year][[date_col, value_col]].copy()
        year_df = year_df.reset_index(drop=True)
        year_dfs[year] = year_df
    
    return year_dfs

def analyze_year_splits(year_dfs: Dict[int, pd.DataFrame], date_col: str = "date_index") -> pd.DataFrame:
    """
    Analyze the year splits to check record counts and date ranges
    
    Args:
        year_dfs: Dictionary of year DataFrames
        date_col: Name of date column
    
    Returns:
        Summary DataFrame with year statistics
    """
    summary_data = []
    
    for year, df in year_dfs.items():
        start_date = df[date_col].min()
        end_date = df[date_col].max()
        record_count = len(df)
        days_covered = (end_date - start_date).days + 1
        hours_per_day = record_count / days_covered if days_covered > 0 else 0
        
        summary_data.append({
            'year': year,
            'records': record_count,
            'start_date': start_date,
            'end_date': end_date,
            'days_covered': days_covered,
            'avg_hours_per_day': round(hours_per_day, 2)
        })
    
    return pd.DataFrame(summary_data)

def create_aligned_year_comparison(year_dfs: Dict[int, pd.DataFrame], 
                                  date_col: str = "date_index",
                                  value_col: Optional[str] = None) -> pd.DataFrame:
    """
    Create aligned comparison preserving hourly structure
    Limits to 8760 hours and uses hour_of_year as explicit column
    
    Args:
        year_dfs: Dictionary of year DataFrames
        date_col: Name of date column
        value_col: Name of value column
    
    Returns:
        DataFrame with hour_of_year column and years as separate columns
    """
    aligned_data = []
    
    for year, df in year_dfs.items():
        df_work = df.copy()
        if value_col is None:
            value_col = df_work.columns[1]
        
        df_work[date_col] = pd.to_datetime(df_work[date_col])
        
        # Create hour of year (1-8760 max)
        year_start = pd.Timestamp(f'{year}-01-01')
        df_work['hour_of_year'] = ((df_work[date_col] - year_start).dt.total_seconds() / 3600).astype(int) + 1
        
        # Filter to max 8760 hours (exclude Feb 29 for leap years)
        df_work = df_work[df_work['hour_of_year'] <= 8760]
        
        # Select only needed columns and rename value column to year
        year_data = df_work[['hour_of_year', value_col]].rename(columns={value_col: value_col+"_"+str(year)})
        aligned_data.append(year_data)
    
    # Merge all years on hour_of_year
    if aligned_data:
        result_df = aligned_data[0]
        for year_data in aligned_data[1:]:
            result_df = result_df.merge(year_data, on='hour_of_year', how='outer')
        result_df = result_df.sort_values('hour_of_year').reset_index(drop=True)
    else:
        result_df = pd.DataFrame()
    
    return result_df

In [7]:
dic_co_anual = split_timeseries_by_years(df = df_co_suroeste,
                                         value_col = 'co')
test_dic_co_anual = analyze_year_splits(year_dfs = dic_co_anual)
test_dic_co_anual

,year,records,start_date,end_date,days_covered,avg_hours_per_day
0,2022,8760,2022-01-01,2022-12-31 23:00:00,365,24.00
1,2023,8758,2023-01-01,2023-12-31 23:00:00,365,23.99
2,2024,8784,2024-01-01,2024-12-31 23:00:00,366,24.00


In [8]:
df_aligned_co = create_aligned_year_comparison(year_dfs = dic_co_anual,
                                               date_col = 'date_index',
                                               value_col= 'co')
df_aligned_co

,hour_of_year,co_2022,co_2023,co_2024
0,1,NaN,3.67,4.21
1,2,3.28,3.90,4.21
2,3,3.25,4.28,4.37
3,4,3.02,3.15,3.35
4,5,2.73,2.32,3.53
...,...,...,...,...
8755,8756,1.65,2.72,4.14
8756,8757,2.04,4.46,4.63
8757,8758,3.02,4.46,4.88
8758,8759,3.72,4.74,4.31


In [9]:
path_save = r"data/interim"
# df_aligned_co.to_csv(path_save+"/tabla_anual_co_sureste.csv", index=False)

## Completion of dataset

Usamos enfoques basados en medias móviles, medias inter anuales, interpolación con splines y bootstrapping. 

In [10]:
from typing import Optional, List
from scipy import interpolate
from sklearn.utils import resample

class TimeSeriesImputer:
    """
    Comprehensive imputation strategies for aligned year comparison DataFrame
    """
    
    def __init__(self, df: pd.DataFrame, hour_col: str = 'hour_of_year'):
        """
        Initialize imputer with aligned DataFrame
        
        Args:
            df: DataFrame from create_aligned_year_comparison
            hour_col: Name of hour column
        """
        self.df = df.copy()
        self.hour_col = hour_col
        self.year_cols = [col for col in df.columns if col != hour_col]
        
    def rolling_mean_imputation(self, column: str, window: int = 24) -> pd.Series:
        """
        Strategy 1: Rolling mean within column (centered window)
        
        Args:
            column: Column name to impute
            window: Window size (default 24 for daily pattern)
        
        Returns:
            Series with imputed values
        """
        series = self.df[column].copy()
        
        # Use centered rolling mean
        rolling_mean = series.rolling(window=window, center=True, min_periods=1).mean()
        
        # Fill NaN values with rolling mean
        imputed = series.fillna(rolling_mean)
        
        return imputed
    
    def cross_year_median_imputation(self, target_col: str, window: int = 24) -> pd.Series:
        """
        Strategy 2: Median of rolling means across years for same hour
        
        Args:
            target_col: Target column to impute
            window: Window size for rolling means
        
        Returns:
            Series with imputed values
        """
        result = self.df[target_col].copy()
        
        # Calculate rolling means for all years
        rolling_means = {}
        for col in self.year_cols:
            rolling_means[col] = self.df[col].rolling(window=window, center=True, min_periods=1).mean()
        
        # For each missing value, get median of all rolling means at that hour
        missing_mask = result.isna()
        
        for idx in self.df[missing_mask].index:
            hour_means = []
            for col in self.year_cols:
                if not pd.isna(rolling_means[col].iloc[idx]):
                    hour_means.append(rolling_means[col].iloc[idx])
            
            if hour_means:
                result.iloc[idx] = np.median(hour_means)
        
        return result
    
    def rolling_median_imputation(self, column: str, window: int = 24) -> pd.Series:
        """
        Strategy 3: Rolling median within column
        
        Args:
            column: Column name to impute
            window: Window size
        
        Returns:
            Series with imputed values
        """
        series = self.df[column].copy()
        
        # Use centered rolling median
        rolling_median = series.rolling(window=window, center=True, min_periods=1).median()
        
        # Fill NaN values with rolling median
        imputed = series.fillna(rolling_median)
        
        return imputed
    
    def hourly_mean_imputation(self, column: str) -> pd.Series:
        """
        Strategy 4: Mean of same hour across all available years
        
        Args:
            column: Column name to impute
        
        Returns:
            Series with imputed values
        """
        result = self.df[column].copy()
        
        # For each hour, calculate mean across all years
        for hour in self.df[self.hour_col].unique():
            hour_mask = self.df[self.hour_col] == hour
            hour_values = []
            
            # Collect all values for this hour from all years
            for col in self.year_cols:
                values = self.df.loc[hour_mask, col].dropna()
                hour_values.extend(values.tolist())
            
            if hour_values:
                hour_mean = np.mean(hour_values)
                # Fill missing values for this hour in target column
                missing_hour_mask = hour_mask & result.isna()
                result.loc[missing_hour_mask] = hour_mean
        
        return result
    
    def polynomial_interpolation(self, column: str, degree: int = 2) -> pd.Series:
        """
        Strategy 5: Polynomial interpolation using 5 nearest points
        
        Args:
            column: Column name to impute
            degree: Polynomial degree
        
        Returns:
            Series with imputed values
        """
        series = self.df[column].copy()
        missing_indices = series[series.isna()].index
        
        for idx in missing_indices:
            # Find 5 nearest non-NaN values (2 before, 2 after, or adjust if at boundaries)
            nearby_indices = []
            search_radius = 1
            
            while len(nearby_indices) < 5 and search_radius < len(series):
                for offset in range(-search_radius, search_radius + 1):
                    check_idx = idx + offset
                    if (0 <= check_idx < len(series) and 
                        check_idx not in nearby_indices and 
                        not pd.isna(series.iloc[check_idx])):
                        nearby_indices.append(check_idx)
                search_radius += 1
            
            if len(nearby_indices) >= max(2, degree + 1):  # Need at least degree+1 points
                nearby_indices = sorted(nearby_indices[:5])  # Take closest 5
                x_vals = np.array(nearby_indices)
                y_vals = series.iloc[nearby_indices].values
                
                # Fit polynomial
                if len(nearby_indices) > degree:
                    poly_coef = np.polyfit(x_vals, y_vals, degree)
                    interpolated_value = np.polyval(poly_coef, idx)
                    series.iloc[idx] = interpolated_value
        
        return series
    
    def bootstrap_imputation(self, column: str, n_bootstrap: int = 100, 
                           hour_window: int = 2) -> pd.Series:
        """
        Strategy 6: Bootstrap using values from same hour of day across years
        
        Args:
            column: Column name to impute
            n_bootstrap: Number of bootstrap samples
            hour_window: Window around target hour (±hours)
        
        Returns:
            Series with imputed values
        """
        result = self.df[column].copy()
        missing_mask = result.isna()
        
        for idx in self.df[missing_mask].index:
            target_hour = self.df.loc[idx, self.hour_col]
            hour_of_day = target_hour % 24
            
            # Collect values from similar hours across all years
            bootstrap_pool = []
            
            for col in self.year_cols:
                # Find hours within window
                for h in range(max(1, hour_of_day - hour_window), 
                              min(25, hour_of_day + hour_window + 1)):
                    hour_candidates = self.df[
                        (self.df[self.hour_col] % 24 == h) & 
                        (~self.df[col].isna())
                    ][col]
                    bootstrap_pool.extend(hour_candidates.tolist())
            
            if bootstrap_pool:
                # Perform bootstrap and take mean
                bootstrap_means = []
                for _ in range(n_bootstrap):
                    sample = resample(bootstrap_pool, n_samples=min(len(bootstrap_pool), 10))
                    bootstrap_means.append(np.mean(sample))
                
                result.iloc[idx] = np.mean(bootstrap_means)
        
        return result
    
    def apply_all_strategies(self, column: str) -> pd.DataFrame:
        """
        Apply all imputation strategies to a column
        
        Args:
            column: Column name to impute
        
        Returns:
            DataFrame with original and all imputed versions
        """
        strategies_results = pd.DataFrame({
            'hour_of_year': self.df[self.hour_col],
            'original': self.df[column],
            'rolling_mean': self.rolling_mean_imputation(column),
            'cross_year_median': self.cross_year_median_imputation(column),
            'rolling_median': self.rolling_median_imputation(column),
            'hourly_mean': self.hourly_mean_imputation(column),
            'polynomial': self.polynomial_interpolation(column),
            'bootstrap': self.bootstrap_imputation(column)
        })
        
        return strategies_results


In [11]:
# Inicializar imputer con tu DataFrame alineado
imputer = TimeSeriesImputer(df_aligned_co)

# Aplicar todas las estrategias a una columna específica
imputed_results = imputer.apply_all_strategies('co_2022')  # o cualquier año

In [12]:
imputed_results.head()

,hour_of_year,original,rolling_mean,cross_year_median,rolling_median,hourly_mean,polynomial,bootstrap
0,1,NaN,2.089091,2.3575,1.98,3.94,3.44,1.62181
1,2,3.28,3.280000,3.2800,3.28,3.28,3.28,3.28000
2,3,3.25,3.250000,3.2500,3.25,3.25,3.25,3.25000
3,4,3.02,3.020000,3.0200,3.02,3.02,3.02,3.02000
4,5,2.73,2.730000,2.7300,2.73,2.73,2.73,2.73000


In [13]:
import matplotlib.pyplot as plt 